# Backend API — How Endpoints Are Built and Work

This notebook walks through the backend layer of the LLM Router & Playground, explaining how each API endpoint is constructed, what schemas drive request/response shapes, and how routers connect to adapters, services, and the database.

**Goal:** After reading this + `frontend/src/api/CLIENT_EXPLAINED.md`, you should be able to trace any request from the browser all the way through the backend and back.

---

## 1. Application Assembly (`main.py`)

Everything starts in `app/main.py`. FastAPI's `lifespan` context manager runs once at startup and teardown.

In [ ]:
# main.py — simplified view of what happens at startup

# 1. Lifespan: runs ONCE when the server starts
@asynccontextmanager
async def lifespan(app):
    # a) Configure logging from settings
    logging.basicConfig(level=settings.log_level)

    # b) Create all DB tables (dev convenience — Alembic in prod)
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

    # c) Initialize provider registry — probes which API keys are set
    init_registry()

    # d) Register built-in tools (calculate, word_count, etc.)
    register_default_tools()

    yield  # app runs here

# 2. Create the FastAPI app
app = FastAPI(title="LLM Router & Playground", version="0.1.0", lifespan=lifespan)

# 3. Add middleware (order matters — last added runs first)
app.add_middleware(CORSMiddleware, allow_origins=settings.cors_origin_list, ...)
app.add_middleware(RequestLoggingMiddleware)  # adds X-Request-ID header + timing logs

# 4. Mount routers
app.include_router(health.router)          # /health, /api/providers/{p}/models
app.include_router(chat.router)            # /api/chat, /api/chat/stream, /api/chat/turn, /api/chat/turn/stream
app.include_router(runs.router)            # /api/runs (CRUD)
app.include_router(conversations.router)   # /api/conversations (CRUD)
app.include_router(tools.router)           # /api/tools

### What happens on every request

```
Incoming HTTP request
   │
   ▼
RequestLoggingMiddleware
   ├─ Generates 8-char request ID
   ├─ Logs: [abc12345] POST /api/chat/turn
   ├─ Calls the next handler
   ├─ Logs: [abc12345] 200 142ms
   └─ Adds X-Request-ID response header
   │
   ▼
CORSMiddleware
   ├─ Checks Origin against cors_origin_list
   └─ Adds Access-Control-* headers
   │
   ▼
Router handler (matched by path + method)
```

---

## 2. Configuration (`config.py`)

All settings come from environment variables (or `.env` file). Pydantic Settings handles parsing.

In [ ]:
# config.py — environment-driven settings

class Settings(BaseSettings):
    database_url: str = "sqlite+aiosqlite:///./llm_router.db"
    cors_origins: str = '["http://localhost:5173"]'
    log_level: str = "INFO"

    # Provider keys — all optional. Adapters auto-disable if empty.
    openai_api_key: str = ""
    anthropic_api_key: str = ""
    google_api_key: str = ""
    mistral_api_key: str = ""
    groq_api_key: str = ""
    together_api_key: str = ""
    azure_openai_api_key: str = ""

    # Local / OpenAI-compatible (Ollama, vLLM, LM Studio)
    local_openai_base_url: str = "http://localhost:11434/v1"

    model_config = {"env_file": ".env"}

# Singleton — cached so env is read once
@lru_cache
def get_settings() -> Settings:
    return Settings()

**Key point:** No API key is required at startup. Each adapter checks its own key in `is_available()` — if missing, the adapter is skipped and that provider simply won't appear in `/health`.

---

## 3. Database Layer

### 3a. Engine & Sessions (`database.py`)

In [ ]:
# database.py — async SQLAlchemy setup

engine = create_async_engine(get_settings().database_url, echo=False, future=True)

async_session_factory = async_sessionmaker(engine, class_=AsyncSession, expire_on_commit=False)

# FastAPI dependency — every route handler gets a session via Depends(get_db)
async def get_db() -> AsyncSession:
    async with async_session_factory() as session:
        yield session  # <-- generator-based dependency

**How route handlers get a DB session:**
```python
@router.get("/runs")
async def list_runs(db: AsyncSession = Depends(get_db)):
    # db is an open AsyncSession, auto-closed after the handler returns
```

FastAPI's `Depends(get_db)` calls the generator, gives the session to the handler, then cleans up. No manual open/close in route code.

### 3b. ORM Models (`models.py`)

Three tables, all using UUID primary keys with `str` type for SQLite compatibility.

In [ ]:
# models.py — the three core tables

class Conversation(Base):
    __tablename__ = "conversations"
    id: str             # UUID PK
    created_at: datetime
    updated_at: datetime  # auto-updates on change
    title: str | None
    provider: str         # e.g. "openai"
    model: str            # e.g. "gpt-4o"
    system_prompt: str | None
    config_json: str | None  # extensibility hook
    # relationships
    messages -> [ConversationMessage]  # ordered by ordinal
    runs -> [Run]


class ConversationMessage(Base):
    __tablename__ = "conversation_messages"
    id: str                # UUID PK
    conversation_id: str   # FK → conversations.id (CASCADE delete)
    run_id: str | None     # FK → runs.id (SET NULL on delete)
    role: str              # system | user | assistant | tool
    content: str
    ordinal: int           # message ordering within conversation
    created_at: datetime
    metadata_json: str | None  # extensibility hook


class Run(Base):
    __tablename__ = "runs"
    id: str                   # UUID PK
    created_at: datetime
    provider: str
    model: str
    request_json: str         # full ChatRequest as JSON
    normalized_response_json: str | None  # NormalizedChatResponse as JSON
    raw_response_json: str | None         # provider's raw response
    status: str               # "pending" | "success" | "error"
    error_message: str | None
    latency_ms: float | None
    prompt_tokens: int | None
    completion_tokens: int | None
    total_tokens: int | None
    tags: str | None
    trace_id: str | None      # groups related runs (agentic)
    parent_run_id: str | None  # parent in the run tree
    conversation_id: str | None  # FK → conversations.id

### Entity Relationship Diagram

```
┌───────────────────┐       ┌────────────────────────┐       ┌──────────────┐
│   Conversation    │       │  ConversationMessage    │       │     Run      │
├───────────────────┤       ├────────────────────────┤       ├──────────────┤
│ id (PK)           │──1:N──│ conversation_id (FK)   │       │ id (PK)      │
│ provider, model   │       │ run_id (FK) ───────────│──N:1──│ provider     │
│ system_prompt     │       │ role, content          │       │ model        │
│ title             │       │ ordinal (ordering)     │       │ status       │
│ config_json       │       │ metadata_json          │       │ latency_ms   │
│ created/updated   │       │ created_at             │       │ *_tokens     │
└───────────────────┘       └────────────────────────┘       │ trace_id     │
        │                                                     │ parent_run_id│
        └──────────────────────────1:N────────────────────────│conversation_id│
                                                              └──────────────┘
```

---

## 4. Pydantic Schemas (`schemas.py`)

Schemas define the shape of every request and response. They sit between the HTTP layer and the business logic — routers deserialize incoming JSON into these models, and serialize them back to JSON for responses.

### 4a. Messages & Chat Request

In [ ]:
# The universal message format used across all providers
class Message(BaseModel):
    role: str               # "system" | "user" | "assistant" | "tool"
    content: str | None     # text content (None for assistant tool-call messages)
    name: str | None        # function name (for tool messages)
    tool_call_id: str | None  # which tool call this responds to
    tool_calls: list[ToolCall] | None  # tool calls the assistant wants to make


# What the frontend sends for a chat request
class ChatRequest(BaseModel):
    provider: str                       # "openai", "anthropic", etc.
    model: str                          # "gpt-4o", "claude-sonnet-4-20250514", etc.
    messages: list[Message]             # conversation history
    temperature: float | None = 0.7
    max_tokens: int = 1024
    tools: list[ToolDefinitionRequest] | None = None  # tool definitions for the LLM
    tool_choice: str | None = None      # "auto", "none", or specific tool name
    provider_options: dict = {}          # pass-through for provider-specific params

### 4b. Normalized Response

Every provider adapter must return this same shape — the router layer never sees provider-specific response formats.

In [ ]:
class UsageInfo(BaseModel):
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    total_tokens: int | None = None


class NormalizedChatResponse(BaseModel):
    output_text: str                    # the assistant's text reply
    finish_reason: str | None           # "stop", "tool_calls", "length", etc.
    provider_response_id: str | None    # provider's unique response ID
    usage: UsageInfo                    # token counts
    tool_calls: list[ToolCall] | None   # any tool calls the LLM wants to make
    raw: dict = {}                      # full provider response for debugging

### 4c. SSE Stream Events

Streaming responses use Server-Sent Events. Four event types form a protocol:

In [ ]:
# Sent as text tokens arrive from the LLM
class StreamDelta(BaseModel):
    type: str = "delta"
    text: str                # a chunk of the response, e.g. "Hello"

# Sent once at the start with provider/model info
class StreamMeta(BaseModel):
    type: str = "meta"
    provider: str
    model: str

# Sent once at the end with the complete response
class StreamFinal(BaseModel):
    type: str = "final"
    response: NormalizedChatResponse
    conversation_id: str | None = None
    run_id: str | None = None

# Sent if something goes wrong
class StreamError(BaseModel):
    type: str = "error"
    message: str

**SSE wire format** — each event is sent as:
```
event: delta
data: {"type": "delta", "text": "Hello"}

event: delta
data: {"type": "delta", "text": " world"}

event: meta
data: {"type": "meta", "provider": "openai", "model": "gpt-4o"}

event: final
data: {"type": "final", "response": {...}, "conversation_id": "uuid", "run_id": "uuid"}
```

The frontend's `useStream.ts` hook parses these events using `ReadableStream` — it doesn't use `client.ts` at all for streaming.

### 4d. Conversation Turn Request

The turn-based endpoints use a different request schema than the legacy `ChatRequest`:

In [ ]:
class ConversationTurnRequest(BaseModel):
    conversation_id: str | None = None  # None = create new conversation
    provider: str
    model: str
    message: str                        # just the user's text — not a full message array
    system_prompt: str | None = None    # only used when creating new conversation
    temperature: float | None = 0.7
    max_tokens: int = 1024
    provider_options: dict = {}
    tool_mode: Literal["off", "auto", "manual"] = "off"
    tool_names: list[str] | None = None  # only used when tool_mode="manual"


class TurnResponse(BaseModel):
    conversation_id: str                # always returned (created if new)
    run_id: str                         # ID of the persisted Run
    response: NormalizedChatResponse
    latency_ms: float

**Key difference from `ChatRequest`:** The turn endpoint only takes a single `message` string. The backend builds the full message array from conversation history + system prompt + sliding window. The frontend doesn't manage message history — the backend does.

---

## 5. Provider Adapter Architecture

Adapters are the core extension point. Each LLM provider (OpenAI, Anthropic, etc.) implements one adapter.

### 5a. The Abstract Base Class

In [ ]:
# adapters/base.py

StreamEvent = StreamDelta | StreamMeta | StreamFinal | StreamError

class ProviderAdapter(ABC):
    name: str  # e.g. "openai", "anthropic"

    @abstractmethod
    def is_available(self) -> bool:
        """Check if credentials are present in environment."""

    @abstractmethod
    async def chat(self, req: ChatRequest) -> NormalizedChatResponse:
        """Non-streaming completion. Returns full response at once."""

    @abstractmethod
    async def stream_chat(self, req: ChatRequest) -> AsyncIterator[StreamEvent]:
        """Streaming completion. Yields delta/meta/final/error events."""

    async def list_models(self) -> list[str]:
        """Optional: return available model IDs."""
        return []

### 5b. The Registry — Auto-Discovery at Startup

In [ ]:
# adapters/registry.py

# All known adapter classes (8 providers)
_ALL_ADAPTERS = [
    OpenAIAdapter, AnthropicAdapter, OpenAICompatibleAdapter,
    GoogleGeminiAdapter, MistralAdapter, GroqAdapter,
    TogetherAdapter, AzureOpenAIAdapter,
]

_registry: dict[str, ProviderAdapter] = {}

def init_registry():
    """Called once at startup. Probes each adapter and registers available ones."""
    for cls in _ALL_ADAPTERS:
        adapter = cls()
        if adapter.is_available():   # checks if API key is set
            _registry[adapter.name] = adapter
            # log: "Provider registered: openai"
        else:
            # log: "Provider skipped (no credentials): google_gemini"
            pass

def get_adapter(name: str) -> ProviderAdapter | None:
    return _registry.get(name)

def list_providers() -> list[str]:
    return list(_registry.keys())

**How routers use adapters:**
```python
# In any chat endpoint:
adapter = get_adapter(req.provider)        # lookup by name
if not adapter:
    raise HTTPException(400, "Provider not available")
response = await adapter.chat(req)          # provider-agnostic call
# response is always NormalizedChatResponse — same shape regardless of provider
```

The router never imports OpenAI, Anthropic, or any provider SDK directly. It only talks to the adapter interface.

---

## 6. Complete Endpoint Reference

### 6a. Health & Discovery

These are the simplest endpoints — no DB, no adapters (for health), just registry queries.

In [ ]:
# routers/health.py

# GET /health — called by frontend on page load
@router.get("/health")
async def healthcheck() -> dict:
    return {
        "status": "ok",
        "available_providers": list_providers(),  # ["openai", "anthropic", ...]
    }

# GET /api/providers/{provider}/models — called when user switches provider dropdown
@router.get("/api/providers/{provider}/models")
async def provider_models(provider: str) -> dict:
    adapter = get_adapter(provider)
    if not adapter:
        raise HTTPException(404, f"Provider '{provider}' not available")
    models = await adapter.list_models()  # calls provider SDK to fetch model list
    return {"provider": provider, "models": models}

**Frontend connection:** `api.health()` → `GET /health`, `api.listModels(provider)` → `GET /api/providers/:provider/models`

### 6b. Tools

Returns the list of server-side tool definitions so the Playground can show tool toggles.

In [ ]:
# routers/tools.py

# GET /api/tools
@router.get("/tools")
async def list_tools() -> dict:
    return {
        "tools": [d.model_dump() for d in tool_registry.list_definitions()],
        # Each tool: {"name": "calculate", "description": "...", "parameters_schema": {...}}
    }

**Frontend connection:** `api.listTools()` → `GET /api/tools`

---

### 6c. Legacy Chat — `POST /api/chat`

The original single-shot endpoint. Frontend sends a complete message array, gets back a full response.

In [ ]:
# routers/chat.py — POST /api/chat

@router.post("/chat")
async def chat(req: ChatRequest, db: AsyncSession = Depends(get_db)) -> dict:
    # 1. Look up the provider adapter
    adapter = get_adapter(req.provider)
    if not adapter:
        raise HTTPException(400, f"Provider '{req.provider}' not available")

    start = time.perf_counter()
    try:
        # 2. If tools are provided, run the tool-calling loop
        if req.tools:
            executor = ToolExecutor(tool_registry)
            response = await executor.execute_with_tools(adapter, req)
        else:
            # 3. Otherwise, direct call to provider
            response = await adapter.chat(req)

        latency = (time.perf_counter() - start) * 1000

        # 4. Persist run record to DB
        run = await _persist_run(db, req, response, None, latency)
        await db.commit()

        # 5. Return response with run ID and latency
        return {"run_id": run.id, "response": response.model_dump(), "latency_ms": latency}

    except NotImplementedError as exc:
        # Provider doesn't support this operation
        await _persist_run(db, req, None, str(exc), latency)  # persist as error
        raise HTTPException(501, str(exc))
    except Exception as exc:
        # Provider call failed
        await _persist_run(db, req, None, str(exc), latency)  # persist as error
        raise HTTPException(502, str(exc))

**Frontend connection:** `api.chat(req)` → `POST /api/chat`

**Important pattern:** Every request — success or failure — is persisted as a `Run`. Errors get `status="error"` with the error message. This means the History page always shows what happened.

### 6d. Legacy Streaming — `POST /api/chat/stream`

Same as `/api/chat` but returns a Server-Sent Events stream instead of a single JSON body.

In [ ]:
# routers/chat.py — POST /api/chat/stream

@router.post("/chat/stream")
async def chat_stream(req: ChatRequest, db = Depends(get_db)) -> StreamingResponse:
    # Tool calling + streaming not supported yet
    if req.tools:
        raise HTTPException(400, "Tool calling is not supported with streaming yet")

    adapter = get_adapter(req.provider)

    async def event_generator():
        final_response = None
        error_msg = None
        try:
            # adapter.stream_chat() yields StreamEvent objects
            async for event in adapter.stream_chat(req):
                if isinstance(event, StreamDelta):
                    yield f"event: delta\ndata: {event.model_dump_json()}\n\n"
                elif isinstance(event, StreamMeta):
                    yield f"event: meta\ndata: {event.model_dump_json()}\n\n"
                elif isinstance(event, StreamFinal):
                    final_response = event.response
                    yield f"event: final\ndata: {event.model_dump_json()}\n\n"
                elif isinstance(event, StreamError):
                    error_msg = event.message
                    yield f"event: error\ndata: {event.model_dump_json()}\n\n"
        finally:
            # Persist run AFTER streaming completes (in the finally block)
            await _persist_run(db, req, final_response, error_msg, latency)
            await db.commit()

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
    )

**Key details:**
- The `event_generator` is an `AsyncIterator[str]` that FastAPI streams to the client
- Each `yield` sends one SSE event immediately — no buffering
- `X-Accel-Buffering: no` tells nginx/reverse proxies not to buffer the stream
- Run persistence happens in `finally` — even if the stream errors out
- Tool calling is explicitly blocked (400) — streaming + tools is not yet implemented

**Frontend connection:** `useStream.ts` opens its own `fetch` to this endpoint (not via `client.ts`)

### 6e. Conversation Turn — `POST /api/chat/turn` (the primary endpoint)

This is the endpoint the Playground UI primarily uses. Unlike `/api/chat`, the frontend sends a single message string and the backend manages the full conversation history.

In [ ]:
# The shared setup function — used by both /chat/turn and /chat/turn/stream

async def _prepare_turn(db, req: ConversationTurnRequest) -> tuple:
    svc = conversation_service

    # 1. Resolve or create conversation
    if req.conversation_id:
        conv = await svc.get_conversation(db, req.conversation_id)
        if not conv: raise HTTPException(404, "Conversation not found")
    else:
        conv = await svc.create_conversation(db, provider=req.provider, model=req.model,
                                             system_prompt=req.system_prompt)

    # 2. Append the user's message to conversation history
    await svc.append_message(db, conversation_id=conv.id, role="user", content=req.message)

    # 3. Build a full ChatRequest from conversation history
    chat_req = await svc.build_chat_request(
        db, conversation=conv, new_user_message=req.message,
        provider=req.provider, model=req.model,
        temperature=req.temperature, max_tokens=req.max_tokens,
    )

    # 4. Resolve tools based on tool_mode
    tool_defs = []
    if req.tool_mode == "auto":
        tool_defs = tool_registry.list_request_schemas()  # all registered tools
    elif req.tool_mode == "manual" and req.tool_names:
        for name in req.tool_names:
            tool = tool_registry.get(name)
            if not tool: raise HTTPException(400, f"Unknown tool: '{name}'")
            tool_defs.append(tool.as_request_schema())

    if tool_defs:
        chat_req = chat_req.model_copy(update={"tools": tool_defs, "tool_choice": "auto"})

    # 5. Get adapter + create trace context
    adapter = get_adapter(req.provider)
    trace = new_trace()  # new trace_id for this interaction
    parent_run_id = await svc.get_last_run_id(db, conv.id)  # chain to previous run

    return conv, chat_req, adapter, trace, parent_run_id

In [ ]:
# POST /api/chat/turn — non-streaming conversation turn

@router.post("/chat/turn", response_model=TurnResponse)
async def chat_turn(req: ConversationTurnRequest, db = Depends(get_db)) -> TurnResponse:
    conv, chat_req, adapter, trace, parent_run_id = await _prepare_turn(db, req)

    start = time.perf_counter()

    # Call LLM (with tool loop if tools are enabled)
    if chat_req.tools:
        executor = ToolExecutor(tool_registry)
        response = await executor.execute_with_tools(adapter, chat_req)
    else:
        response = await adapter.chat(chat_req)

    latency = (time.perf_counter() - start) * 1000

    # Persist run with trace info
    run = await _persist_run(
        db, chat_req, response, None, latency,
        trace_id=trace.trace_id,
        parent_run_id=parent_run_id,
        conversation_id=conv.id,
    )

    # Append assistant's reply to conversation
    await svc.append_message(
        db, conversation_id=conv.id, role="assistant",
        content=response.output_text, run_id=run.id,
    )
    await db.commit()

    return TurnResponse(
        conversation_id=conv.id,
        run_id=run.id,
        response=response,
        latency_ms=latency,
    )

**Frontend connection:** `api.chatTurn(req)` → `POST /api/chat/turn`

**Complete request flow:**

```
Frontend: api.chatTurn({provider: "openai", model: "gpt-4o", message: "Hello!"})
    │
    ▼  POST /api/chat/turn  (JSON body)
    │
RequestLoggingMiddleware  →  logs [abc12345] POST /api/chat/turn
    │
    ▼
chat_turn() handler
    │
    ├─ _prepare_turn()
    │   ├─ Create or fetch Conversation from DB
    │   ├─ Append user message to conversation_messages table
    │   ├─ build_chat_request():
    │   │   ├─ Prepend system prompt (if any)
    │   │   ├─ Load conversation history from DB
    │   │   ├─ Apply sliding window (keep last 50 messages)
    │   │   ├─ Append new user message
    │   │   └─ Return ChatRequest with full message array
    │   ├─ Resolve tools (off/auto/manual)
    │   ├─ Get adapter from registry
    │   └─ Create TraceContext + find parent_run_id
    │
    ├─ adapter.chat(chat_req)  →  calls OpenAI / Anthropic / etc.
    │   └─ Returns NormalizedChatResponse
    │
    ├─ _persist_run()  →  INSERT into runs table
    ├─ append_message(role="assistant")  →  INSERT into conversation_messages
    ├─ db.commit()
    │
    └─ Return TurnResponse JSON
         │
         ▼
Frontend receives: {conversation_id, run_id, response: {output_text, ...}, latency_ms}
```

### 6f. Streaming Conversation Turn — `POST /api/chat/turn/stream`

Same flow as `/api/chat/turn` but returns SSE instead of a JSON body. Tool calling is blocked (400).

In [ ]:
# POST /api/chat/turn/stream

@router.post("/chat/turn/stream")
async def chat_turn_stream(req: ConversationTurnRequest, db = Depends(get_db)):
    # Block tool calling with streaming
    if req.tool_mode == "auto" or (req.tool_mode == "manual" and req.tool_names):
        raise HTTPException(400, "Tool calling is not supported with streaming")

    conv, chat_req, adapter, trace, parent_run_id = await _prepare_turn(db, req)

    async def event_generator():
        collected_text = ""  # accumulates delta text for persistence
        try:
            async for event in adapter.stream_chat(chat_req):
                if isinstance(event, StreamDelta):
                    collected_text += event.text
                    yield f"event: delta\ndata: {event.model_dump_json()}\n\n"
                elif isinstance(event, StreamFinal):
                    # Inject conversation_id into the final event
                    final_with_ids = StreamFinal(
                        response=event.response, conversation_id=conv.id)
                    yield f"event: final\ndata: {final_with_ids.model_dump_json()}\n\n"
                # ... meta and error events too
        finally:
            # Persist run + assistant message AFTER stream completes
            run = await _persist_run(db, chat_req, final_response, error_msg, latency, ...)
            await svc.append_message(db, conversation_id=conv.id, role="assistant",
                                     content=final_response.output_text or collected_text,
                                     run_id=run.id)
            await db.commit()

    return StreamingResponse(event_generator(), media_type="text/event-stream")

**Frontend connection:** `useStream.ts` opens a `fetch` to `/api/chat/turn/stream` and processes events via `ReadableStream`

**Key difference from legacy `/api/chat/stream`:** This endpoint also injects `conversation_id` into the `final` event, and persists both the Run and the assistant message to the conversation.

---

### 6g. Runs CRUD — `/api/runs`

In [ ]:
# routers/runs.py

# GET /api/runs?page=1&per_page=20 — paginated list
@router.get("/runs", response_model=PaginatedRuns)
async def list_runs(page=1, per_page=20, db=Depends(get_db)):
    total = await db.execute(select(func.count(Run.id)))  # count query
    runs = await db.execute(
        select(Run).order_by(Run.created_at.desc())
        .offset((page-1) * per_page).limit(per_page)
    )
    return PaginatedRuns(
        items=[RunSummary.model_validate(r) for r in runs],
        total=total, page=page, per_page=per_page,
    )

# GET /api/runs/{run_id} — full run detail
@router.get("/runs/{run_id}", response_model=RunDetail)
async def get_run(run_id: str, db=Depends(get_db)):
    run = ...  # select by ID
    if not run: raise HTTPException(404, "Run not found")
    return RunDetail.model_validate(run)

# DELETE /api/runs/{run_id}
@router.delete("/runs/{run_id}")
async def delete_run(run_id: str, db=Depends(get_db)):
    run = ...  # select by ID
    if not run: raise HTTPException(404, "Run not found")
    await db.delete(run)
    await db.commit()
    return {"deleted": run_id}

**Frontend connection:**
- `api.listRuns(page, perPage)` → `GET /api/runs?page=&per_page=`
- `api.getRun(id)` → `GET /api/runs/:id`
- `api.deleteRun(id)` → `DELETE /api/runs/:id`

### 6h. Conversations CRUD — `/api/conversations`

In [ ]:
# routers/conversations.py

# GET /api/conversations?page=1&per_page=20 — paginated list with message counts
@router.get("/conversations", response_model=PaginatedConversations)
async def list_conversations(page=1, per_page=20, db=Depends(get_db)):
    # Uses a subquery to efficiently count messages per conversation
    msg_count_sq = (
        select(ConversationMessage.conversation_id,
               func.count(ConversationMessage.id).label("msg_count"))
        .group_by(ConversationMessage.conversation_id)
        .subquery()
    )
    # LEFT JOIN to get conversations even if they have 0 messages
    q = select(Conversation, func.coalesce(msg_count_sq.c.msg_count, 0))
        .outerjoin(msg_count_sq, ...)
        .order_by(Conversation.updated_at.desc())
    ...

# GET /api/conversations/{id} — full detail with all messages
@router.get("/conversations/{conversation_id}", response_model=ConversationDetail)
async def get_conversation(conversation_id, db=Depends(get_db)):
    conv = await conversation_service.get_conversation(db, conversation_id)
    # get_conversation uses selectinload(Conversation.messages) to eager-load
    return ConversationDetail(
        ...,
        messages=[ConversationMessageSchema.model_validate(m) for m in conv.messages],
    )

# PATCH /api/conversations/{id} — rename
@router.patch("/conversations/{conversation_id}")
async def update_conversation(conversation_id, body: dict, db=Depends(get_db)):
    conv = ...
    if "title" in body:
        conv.title = body["title"]
    await db.commit()
    return {"id": conv.id, "title": conv.title}

# DELETE /api/conversations/{id}
@router.delete("/conversations/{conversation_id}")
async def delete_conversation(conversation_id, db=Depends(get_db)):
    # CASCADE delete removes all conversation_messages too
    await db.delete(conv)
    await db.commit()
    return {"deleted": conversation_id}

**Frontend connection:**
- `api.listConversations(page, perPage)` → `GET /api/conversations?page=&per_page=`
- `api.getConversation(id)` → `GET /api/conversations/:id`
- `api.updateConversation(id, {title})` → `PATCH /api/conversations/:id`
- `api.deleteConversation(id)` → `DELETE /api/conversations/:id`

---

## 7. Service Layer

### 7a. ConversationService (`services/conversation.py`)

The service layer owns conversation state management. Routers call it — they never directly query conversation tables.

In [ ]:
# services/conversation.py

class ConversationService:
    def __init__(self, memory_store=None):
        self._memory_store = memory_store  # optional: for memory retrieval (Approach 3)

    async def create_conversation(self, db, *, provider, model, system_prompt=None, title=None):
        """Create a new Conversation row."""

    async def get_conversation(self, db, conversation_id):
        """Fetch conversation with eager-loaded messages."""

    async def append_message(self, db, *, conversation_id, role, content, run_id=None):
        """Add a message with auto-incrementing ordinal."""
        # ordinal = MAX(ordinal) + 1 for this conversation

    async def build_chat_request(self, db, *, conversation, new_user_message,
                                 provider=None, model=None, temperature=0.7,
                                 max_tokens=1024, window_size=50):
        """
        THE KEY METHOD — assembles a ChatRequest from conversation state.

        Steps:
        1. (Optional) Retrieve memories from MemoryStore and prepend as system messages
        2. Prepend the conversation's system_prompt
        3. Load all conversation messages from DB
        4. Apply sliding window (keep last 50 messages)
        5. Append the new user message
        6. Return a ChatRequest ready to send to any adapter
        """

    async def get_last_run_id(self, db, conversation_id):
        """Get the most recent assistant message's run_id for parent_run_id chaining."""

# Singleton — shared across all requests
conversation_service = ConversationService()

**The sliding window** is critical for long conversations. Without it, token counts would grow without bound:

```
Full conversation:  [sys] [u1] [a1] [u2] [a2] ... [u50] [a50] [u51] [a51]
                                                          ─────────────────
Sliding window=50:  keeps only the last 50 messages  ────────────────────┘
                                        
Sent to LLM:  [system_prompt] + [last 50 messages] + [new user message]
```

### 7b. ToolExecutor (`services/tool_executor.py`)

Implements the agentic tool-calling loop — the LLM can call tools, see results, and decide what to do next.

In [ ]:
# services/tool_executor.py

class ToolExecutor:
    def __init__(self, registry: ToolRegistry, max_iterations=10):
        self._registry = registry
        self._max_iterations = max_iterations

    async def execute_with_tools(self, adapter, req: ChatRequest) -> NormalizedChatResponse:
        messages = list(req.messages)  # mutable copy
        total_usage = UsageInfo(prompt_tokens=0, completion_tokens=0, total_tokens=0)

        for i in range(self._max_iterations):
            # 1. Call the LLM with current messages + tool definitions
            current_req = req.model_copy(update={"messages": messages})
            response = await adapter.chat(current_req)
            _accumulate_usage(total_usage, response.usage)

            # 2. If no tool calls, we're done — LLM gave a final text response
            if not response.tool_calls:
                break

            # 3. LLM wants to call tools — execute each one
            messages.append(Message(
                role="assistant", content=response.output_text or None,
                tool_calls=response.tool_calls,
            ))

            for tc in response.tool_calls:
                result = await self._execute_single(tc.function.name, tc.function.arguments)
                messages.append(Message(
                    role="tool", content=result, tool_call_id=tc.id,
                ))

            # 4. Loop back — send messages (now including tool results) to LLM again

        response.usage = total_usage  # aggregate token counts across all iterations
        return response

**The tool-calling loop visualized:**

```
Iteration 1:
  Messages: [system, user: "what time is it and what's 2+2?"]
  LLM returns: tool_calls=[get_current_time(), calculate("2+2")]
  Execute tools → append results

Iteration 2:
  Messages: [system, user, assistant(tool_calls), tool(time_result), tool(calc_result)]
  LLM returns: output_text="It's 3:42 PM and 2+2=4", tool_calls=None
  No more tool calls → break

Return final response (with aggregated token usage from both iterations)
```

The `max_iterations=10` safety limit prevents infinite loops if the LLM keeps calling tools.

---

## 8. Tool System (`agentic/tools.py`)

### 8a. Tool Abstraction

In [ ]:
# agentic/tools.py

class ToolDefinition(BaseModel):
    name: str                    # "calculate"
    description: str             # "Evaluates a math expression..."
    parameters_schema: dict      # JSON Schema for the tool's arguments

class Tool(ABC):
    definition: ToolDefinition

    @abstractmethod
    async def execute(self, arguments: dict) -> str:
        """Run the tool, return string result."""

    def as_request_schema(self) -> ToolDefinitionRequest:
        """Convert to the wire format that LLM APIs expect."""
        # Returns: {"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}


class ToolRegistry:
    def register(self, tool: Tool): ...
    def get(self, name: str) -> Tool | None: ...
    def list_definitions(self) -> list[ToolDefinition]: ...        # for GET /api/tools
    def list_request_schemas(self) -> list[ToolDefinitionRequest]: # for LLM requests

### 8b. Built-in Tools

| Tool | Description | Parameters |
|------|-------------|------------|
| `get_current_time` | Returns current ISO 8601 datetime | None |
| `calculate` | Safe math eval (AST-based, no `eval()`) | `expression: str` |
| `generate_uuid` | Random UUID v4 | None |
| `word_count` | Counts words, chars, sentences, lines | `text: str` |
| `json_formatter` | Pretty-print, minify, or validate JSON | `json_string: str, mode: str` |

All tools are registered at startup via `register_default_tools()` in `main.py`'s lifespan.

### 8c. How Tools Flow Between Frontend and Backend

```
┌─ Frontend (Playground.tsx) ─────────────────────────────────────────────┐
│                                                                         │
│  1. On mount: api.listTools() → GET /api/tools                         │
│     Receives: [{name, description, parameters_schema}, ...]            │
│     Renders: tool toggle switches in the UI                            │
│                                                                         │
│  2. User sends message with tool_mode="auto" (or "manual"):            │
│     api.chatTurn({message: "...", tool_mode: "auto", ...})             │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─ Backend (chat.py → _prepare_turn) ────────────────────────────────────┐
│                                                                         │
│  3. Resolve tools:                                                      │
│     tool_mode="auto" → tool_registry.list_request_schemas()            │
│     tool_mode="manual" → tool_registry.get(name) for each tool_name    │
│                                                                         │
│  4. Attach tool definitions to ChatRequest:                            │
│     chat_req.tools = [{type: "function", function: {name, desc, ...}}] │
│     chat_req.tool_choice = "auto"                                      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
┌─ ToolExecutor.execute_with_tools() ────────────────────────────────────┐
│                                                                         │
│  5. Send to LLM adapter with tool definitions                          │
│  6. LLM returns tool_calls → execute tools → append results            │
│  7. Re-send to LLM → repeat until final text response                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## 9. Run Persistence (`_persist_run`)

Every chat request — streaming or not, success or failure — is saved as a Run.

In [ ]:
# routers/chat.py — helper used by ALL chat endpoints

async def _persist_run(
    db, req: ChatRequest, response: NormalizedChatResponse | None,
    error: str | None, latency_ms: float, *,
    trace_id=None, parent_run_id=None, conversation_id=None,
) -> Run:
    run = Run(
        provider=req.provider,
        model=req.model,
        request_json=req.model_dump_json(),                # full request as JSON string
        normalized_response_json=response.model_dump_json() if response else None,
        raw_response_json=json.dumps(response.raw) if response else None,
        status="success" if response else "error",
        error_message=error,
        latency_ms=latency_ms,
        prompt_tokens=response.usage.prompt_tokens if response else None,
        completion_tokens=response.usage.completion_tokens if response else None,
        total_tokens=response.usage.total_tokens if response else None,
        trace_id=trace_id,
        parent_run_id=parent_run_id,
        conversation_id=conversation_id,
    )
    db.add(run)
    await db.flush()  # flush to get the auto-generated ID
    return run

**Key design:** `db.flush()` (not `commit()`) is called here. The caller commits. This lets the turn endpoints do additional work (append assistant message) before committing everything in one transaction.

---

## 10. Tracing (`agentic/traces.py`)

Traces group related Runs together — useful for multi-step agentic workflows.

In [ ]:
# agentic/traces.py

@dataclass
class TraceContext:
    trace_id: str = field(default_factory=lambda: str(uuid.uuid4()))
    parent_run_id: str | None = None

    def child(self, run_id: str) -> TraceContext:
        """Create a child context with the same trace_id but a new parent."""
        return TraceContext(trace_id=self.trace_id, parent_run_id=run_id)

def new_trace() -> TraceContext:
    return TraceContext()

**How traces work in `/api/chat/turn`:**

```
Turn 1:  trace_id=aaa, parent_run_id=None  →  creates Run(id=r1, trace_id=aaa)
Turn 2:  trace_id=bbb, parent_run_id=r1    →  creates Run(id=r2, trace_id=bbb, parent=r1)
Turn 3:  trace_id=ccc, parent_run_id=r2    →  creates Run(id=r3, trace_id=ccc, parent=r2)
```

The `parent_run_id` chain lets you reconstruct the conversation's execution history from the `runs` table.

---

## 11. Complete Endpoint Map

| Method | Path | Router | Frontend `api.*` Method | Description |
|--------|------|--------|------------------------|-------------|
| GET | `/health` | health | `health()` | Server status + available providers |
| GET | `/api/providers/{p}/models` | health | `listModels(p)` | Models for a provider |
| GET | `/api/tools` | tools | `listTools()` | All tool definitions |
| POST | `/api/chat` | chat | `chat(req)` | Legacy single-shot chat |
| POST | `/api/chat/stream` | chat | *(useStream.ts)* | Legacy streaming chat |
| POST | `/api/chat/turn` | chat | `chatTurn(req)` | Conversation turn (non-streaming) |
| POST | `/api/chat/turn/stream` | chat | *(useStream.ts)* | Conversation turn (streaming) |
| GET | `/api/runs` | runs | `listRuns(page, perPage)` | Paginated run list |
| GET | `/api/runs/{id}` | runs | `getRun(id)` | Run detail |
| DELETE | `/api/runs/{id}` | runs | `deleteRun(id)` | Delete run |
| GET | `/api/conversations` | conversations | `listConversations(page, perPage)` | Paginated conversation list |
| GET | `/api/conversations/{id}` | conversations | `getConversation(id)` | Conversation detail + messages |
| PATCH | `/api/conversations/{id}` | conversations | `updateConversation(id, body)` | Rename conversation |
| DELETE | `/api/conversations/{id}` | conversations | `deleteConversation(id)` | Delete conversation + messages |

---

## 12. End-to-End: Connecting Frontend to Backend

Here's how the full stack connects, combining what you know from `CLIENT_EXPLAINED.md` with this notebook:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  FRONTEND (localhost:5173)                                                  │
│                                                                             │
│  Playground.tsx                                                              │
│    ├─ On mount:                                                              │
│    │   ├─ api.health()               → GET /health                          │
│    │   ├─ api.listTools()            → GET /api/tools                       │
│    │   └─ api.listModels(provider)   → GET /api/providers/:p/models         │
│    │                                                                         │
│    ├─ User sends message (streaming OFF, tools OFF):                        │
│    │   └─ api.chatTurn(req)          → POST /api/chat/turn                  │
│    │       Returns: {conversation_id, run_id, response, latency_ms}         │
│    │                                                                         │
│    ├─ User sends message (streaming ON, tools OFF):                         │
│    │   └─ useStream.ts               → POST /api/chat/turn/stream           │
│    │       Events: delta → delta → ... → meta → final                       │
│    │                                                                         │
│    └─ User sends message (tools ON, streaming OFF):                         │
│        └─ api.chatTurn(req)          → POST /api/chat/turn                  │
│            Backend runs ToolExecutor loop internally                         │
│                                                                             │
│  History.tsx                                                                 │
│    ├─ api.listRuns()                 → GET /api/runs                        │
│    ├─ api.listConversations()        → GET /api/conversations               │
│    └─ api.deleteRun(id)              → DELETE /api/runs/:id                 │
│                                                                             │
│  RunDetail.tsx                                                               │
│    └─ api.getRun(id)                 → GET /api/runs/:id                    │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
          │                        │                         │
          │   Vite dev proxy       │   (forwards /api/*)     │
          ▼                        ▼                         ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│  BACKEND (localhost:8000)                                                   │
│                                                                             │
│  Middleware: RequestLogging → CORS                                           │
│       │                                                                     │
│       ▼                                                                     │
│  Routers: health.py  chat.py  runs.py  conversations.py  tools.py          │
│       │                  │                                                   │
│       │                  ▼                                                   │
│       │        ConversationService  (build history, manage state)            │
│       │                  │                                                   │
│       │                  ▼                                                   │
│       │        ToolExecutor  (tool-calling loop, if tools enabled)           │
│       │                  │                                                   │
│       ▼                  ▼                                                   │
│  Adapter Registry → ProviderAdapter.chat() / stream_chat()                  │
│       │                  │                                                   │
│       │                  ▼                                                   │
│       │         OpenAI / Anthropic / Ollama / etc.                           │
│       │                                                                     │
│       ▼                                                                     │
│  SQLite (async) ← _persist_run() ← append_message()                        │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## 13. Key Design Decisions

| Decision | Rationale |
|----------|----------|
| **Normalized responses** | Routers never see provider-specific shapes. Switching providers is a one-field change. |
| **Adapter auto-discovery** | No manual enable/disable. Set an API key → provider appears. Remove it → provider disappears. |
| **Thin routers** | Route handlers validate input, delegate to services/adapters, shape HTTP response. No business logic in routes. |
| **Async everything** | All DB queries, provider calls, and streaming use `async/await`. No blocking calls anywhere. |
| **Flush vs Commit** | `_persist_run` flushes (to get the ID). Caller commits. This keeps transactions atomic across multiple operations. |
| **Run = audit log** | Every request is persisted, even errors. The History page is a complete record of all LLM interactions. |
| **Conversation ≠ ChatRequest** | `ConversationTurnRequest` takes a single message. `ConversationService.build_chat_request()` assembles the full message array from DB state. Frontend doesn't manage history. |
| **Streaming bypasses client.ts** | SSE streams can't use the JSON request/response pattern. `useStream.ts` does its own fetch + ReadableStream parsing. |
| **Tool calling blocks streaming** | Tools need to see the full response before deciding next steps. Until streaming + tools is implemented, the backend returns 400. |